# 🛰️ SkyGuard AI — Training Notebook (SIH26073, MoES/IMD)

Offline anomaly-detection training for Automatic Weather Stations (T, P, RH).
**Requires** the fresh output of the online data-builder notebook attached via
*Add Data*. Do not attach the source-only bootstrap dataset. Everything here
runs with Kaggle's preinstalled packages only.

Headline metrics are trained on NOAA ISD station observations only. Open-Meteo
is a co-located reanalysis reference and is reported separately; it is never
presented as an AWS fault-label source.

Pipeline: physics QC → robust per-channel detectors → Isolation Forest →
LSTM-AE (24h & 168h) → Transformer-AE → LSTM forecaster → spatial buddy-check
→ calibrated ensemble → per-station thresholds → rigorous evaluation.

## 0 · Bootstrap (offline-safe)

In [ ]:
import os, sys, glob, zipfile, json, time, pathlib, random, shutil, hashlib
import numpy as np, pandas as pd

WORKDIR = "/kaggle/working"
WORK_BUNDLE = os.path.join(WORKDIR, "awsad_bundle")
EXPECTED_ROLE = "processed_training_bundle"
EXPECTED_PARSER = "isd-qc-anomaly-labels-v3"

def _manifest_ok(blob):
    return (
        isinstance(blob, dict)
        and blob.get("bundle_role") == EXPECTED_ROLE
        and blob.get("parser_version") == EXPECTED_PARSER
        and bool(blob.get("build_id"))
    )

def locate_bundles():
    candidates = []
    roots = {
        str(pathlib.Path(p).resolve().parents[2])
        for p in glob.glob("/kaggle/input/**/src/awsad/__init__.py", recursive=True)
    }
    for root in sorted(roots):
        manifest_path = os.path.join(root, "MANIFEST.json")
        if not os.path.isfile(manifest_path):
            continue
        try:
            blob = json.loads(pathlib.Path(manifest_path).read_text(encoding="utf-8"))
        except Exception:
            continue
        if _manifest_ok(blob):
            candidates.append({
                "kind": "root", "path": root, "prefix": "",
                "manifest": blob,
            })

    for archive in sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True)):
        try:
            with zipfile.ZipFile(archive) as zf:
                for name in zf.namelist():
                    if not name.endswith("MANIFEST.json"):
                        continue
                    try:
                        blob = json.loads(zf.read(name).decode("utf-8"))
                    except Exception:
                        continue
                    if _manifest_ok(blob):
                        candidates.append({
                            "kind": "zip", "path": archive,
                            "prefix": name[:-len("MANIFEST.json")],
                            "manifest": blob,
                        })
        except zipfile.BadZipFile:
            continue
    return candidates

candidates = locate_bundles()
if not candidates:
    raise FileNotFoundError(
        "Attach the saved output of the online data-builder notebook. "
        "The source-only bootstrap dataset is not valid training input."
    )
build_ids = {str(c["manifest"]["build_id"]) for c in candidates}
if len(build_ids) != 1:
    raise RuntimeError(
        "Multiple processed bundle builds are attached. Remove old inputs and "
        "attach exactly one builder output. Found build IDs: " + repr(sorted(build_ids))
    )

# Prefer the unzipped notebook output when Kaggle exposes both the output
# directory and its downloadable ZIP for the same build.
chosen = sorted(candidates, key=lambda c: (c["kind"] != "root", c["path"]))[0]
if os.path.isdir(WORK_BUNDLE):
    shutil.rmtree(WORK_BUNDLE)
if chosen["kind"] == "root":
    BUNDLE = chosen["path"]
    print("using processed bundle:", BUNDLE)
else:
    with zipfile.ZipFile(chosen["path"]) as zf:
        zf.extractall(WORK_BUNDLE)
    BUNDLE = os.path.join(WORK_BUNDLE, chosen["prefix"].rstrip("/"))
    print("extracted processed bundle:", chosen["path"])
BUNDLE_MANIFEST = chosen["manifest"]

required_bundle = [
    "src/awsad/__init__.py",
    "data/processed/train_clean.parquet",
    "data/processed/val_labeled.parquet",
    "data/processed/test_labeled.parquet",
    "data/processed/injection_events.parquet",
    "data/processed/splits.json",
    "data/processed/station_stats.json",
    "data/processed/station_catalog.json",
    "data/processed/source_metadata.json",
]
missing = [rel for rel in required_bundle
           if not os.path.isfile(os.path.join(BUNDLE, *rel.split("/")))]
if missing:
    raise RuntimeError("processed bundle is incomplete: " + repr(missing))

hashes = BUNDLE_MANIFEST.get("sha256")
if not isinstance(hashes, dict) or not hashes:
    raise RuntimeError("processed bundle manifest has no content hashes")
bundle_root = pathlib.Path(BUNDLE).resolve()
for rel, expected in hashes.items():
    expected = str(expected).lower()
    if len(expected) != 64 or any(ch not in "0123456789abcdef" for ch in expected):
        raise RuntimeError("invalid SHA-256 in processed manifest: " + repr(rel))
    rel_path = pathlib.PurePosixPath(str(rel))
    if rel_path.is_absolute() or ".." in rel_path.parts:
        raise RuntimeError("unsafe path in processed manifest: " + repr(rel))
    path = bundle_root.joinpath(*rel_path.parts).resolve()
    if bundle_root != path and bundle_root not in path.parents:
        raise RuntimeError("processed manifest path escapes bundle: " + repr(rel))
    if not path.is_file():
        raise RuntimeError("manifest file is missing: " + str(rel))
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    if h.hexdigest() != expected:
        raise RuntimeError("bundle hash mismatch: " + str(rel))
actual_hashed_files = {
    path.relative_to(BUNDLE).as_posix()
    for path in pathlib.Path(BUNDLE).rglob("*")
    if path.is_file() and path.name != "MANIFEST.json"
}
if actual_hashed_files != set(hashes):
    raise RuntimeError(
        "processed bundle manifest coverage mismatch; missing="
        + repr(sorted(actual_hashed_files - set(hashes))[:20])
        + " unexpected="
        + repr(sorted(set(hashes) - actual_hashed_files)[:20])
    )
print("verified bundle build:", BUNDLE_MANIFEST["build_id"],
      "| files:", len(hashes))

src_dir = os.path.join(BUNDLE, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
PROCESSED = os.path.join(BUNDLE, "data", "processed")
NAB_DIR   = os.path.join(BUNDLE, "nab")
OUT       = os.path.join(WORKDIR, "artifacts")
if os.path.isdir(OUT):
    shutil.rmtree(OUT)
os.makedirs(OUT, exist_ok=True)

import torch
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    # RTX Pro 6000-class Tensor Cores; this keeps the large-GPU run practical.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
GPU_GB = (torch.cuda.get_device_properties(0).total_memory / 2**30
          if DEVICE == "cuda" else 0.0)
print("torch", torch.__version__, "| device:", DEVICE,
      "| GPU:", GPU_NAME, f"({GPU_GB:.1f} GiB)")
if DEVICE != "cuda":
    raise RuntimeError("A CUDA GPU is required for the headline run. Enable the Kaggle GPU accelerator before Run All.")
if GPU_GB < 80.0:
    raise RuntimeError(
        f"The committed high-capacity configuration requires at least 80 GiB VRAM; "
        f"this accelerator exposes {GPU_GB:.1f} GiB. Select RTX Pro 6000 96 GB."
    )


## 1 · Dataset sanity

In [ ]:
from awsad.train_pipeline import load_processed, split_groups

dfs = load_processed(PROCESSED)
groups = split_groups(dfs)
split_meta = dfs.get("splits", {})
source_meta_path = os.path.join(PROCESSED, "source_metadata.json")
if not os.path.exists(source_meta_path):
    raise RuntimeError("source_metadata.json is missing; rebuild the online bundle before training")
source_meta = json.loads(pathlib.Path(source_meta_path).read_text(encoding="utf-8"))
assert source_meta.get("parser_version") == "isd-qc-anomaly-labels-v3", \
    "bundle was built with a stale ISD parser; refuse headline training"
BUILD_ID = str(BUNDLE_MANIFEST["build_id"])
for name, blob in (("source_metadata", source_meta), ("splits", split_meta)):
    assert str(blob.get("build_id")) == BUILD_ID, (
        f"{name} build_id does not match MANIFEST"
    )
    assert blob.get("parser_version") == EXPECTED_PARSER, (
        f"{name} has a stale parser version"
    )
assert source_meta.get("status") == "fresh_online_builder", (
    "processed data is not marked as a fresh online builder output"
)

required = {"timestamp", "station_id", "source", "temperature_c",
            "pressure_hpa", "relative_humidity_pct"}
for split_name in ("train", "val", "test"):
    missing = sorted(required - set(dfs[split_name].columns))
    assert not missing, f"{split_name} is missing required columns: {missing}"
assert "label" not in dfs["train"] or int(dfs["train"]["label"].sum()) == 0, \
    "train_clean must not contain injected anomaly labels"
def group_keys(frame):
    return sorted({
        f"{sid}|{src}"
        for sid, src in zip(frame["station_id"].astype(str),
                            frame["source"].astype(str))
    })

bounds = {
    "train": (None, pd.Timestamp("2023-01-01", tz="UTC")),
    "val": (pd.Timestamp("2023-01-01", tz="UTC"),
            pd.Timestamp("2024-01-01", tz="UTC")),
    "test": (pd.Timestamp("2024-01-01", tz="UTC"),
             pd.Timestamp("2026-02-01", tz="UTC")),
}
actual_groups = {}
for split_name in ("train", "val", "test"):
    frame = dfs[split_name]
    ts = pd.to_datetime(frame["timestamp"], utc=True, errors="coerce")
    assert ts.notna().all(), f"{split_name} contains invalid timestamps"
    lo, hi = bounds[split_name]
    if lo is not None:
        assert bool((ts >= lo).all()), f"{split_name} starts before {lo}"
    assert bool((ts < hi).all()), f"{split_name} extends beyond {hi}"
    actual_groups[split_name] = group_keys(frame)

for split_name in ("val", "test"):
    assert "label" in dfs[split_name], f"{split_name} is missing labels"
    labels = pd.to_numeric(dfs[split_name]["label"], errors="coerce")
    assert labels.notna().all(), f"{split_name} contains invalid labels"
    assert set(labels.astype(int).unique()) <= {0, 1}, (
        f"{split_name} labels must be binary"
    )

declared_groups = split_meta.get("persisted_groups_by_split")
assert isinstance(declared_groups, dict), (
    "splits.json lacks persisted_groups_by_split; rebuild with the current builder"
)
for split_name in ("train", "val", "test"):
    assert sorted(declared_groups.get(split_name, [])) == actual_groups[split_name], (
        f"{split_name} group metadata does not match its Parquet"
    )

assert len(groups) > 0, "no station|source groups were found"
NOAA_GROUPS = sorted(split_meta.get("eligible_headline_groups", []))
OPENMETEO_GROUPS = sorted(g for g in groups if g.endswith("|openmeteo"))
manifest_min_groups = int(BUNDLE_MANIFEST.get(
    "validation_contract", {}
).get("minimum_eligible_headline_groups", 0))
assert manifest_min_groups >= 250, (
    "processed bundle weakens the required NOAA network-size contract"
)
assert len(NOAA_GROUPS) >= manifest_min_groups, (
    f"only {len(NOAA_GROUPS)} eligible NOAA groups; "
    f"the manifest requires {manifest_min_groups}"
)
assert all(g.endswith("|noaa_isd") for g in NOAA_GROUPS), (
    "headline groups must be NOAA observation streams"
)
for group in NOAA_GROUPS:
    assert group in groups, f"headline group is absent from train data: {group}"
    sid, src = group.split("|", 1)
    for split_name in ("train", "val", "test"):
        frame = dfs[split_name]
        sub = frame[(frame.station_id.astype(str) == sid) &
                    (frame.source.astype(str) == src)]
        assert len(sub) >= 2000, f"{group} has insufficient {split_name} rows"
        for channel in ("temperature_c", "pressure_hpa", "relative_humidity_pct"):
            finite = np.isfinite(pd.to_numeric(sub[channel], errors="coerce"))
            assert int(finite.sum()) >= 2000, (
                f"{group} has insufficient finite {channel} in {split_name}"
            )

holdout_stations = {str(s) for s in split_meta.get("holdout_stations", [])}
eligible_stations = {g.split("|", 1)[0] for g in NOAA_GROUPS}
assert holdout_stations and holdout_stations < eligible_stations, (
    "holdout set must be nonempty and leave non-holdout fit stations"
)
print(f"{len(groups)} station|source groups ({len(NOAA_GROUPS)} NOAA, {len(OPENMETEO_GROUPS)} Open-Meteo reference)")
print(f"stations: {split_meta.get('station_count', 'unknown')} | "
      f"holdout stations: {len(split_meta.get('holdout_stations', []))} | "
      f"holdout groups are selected from splits.json")
for s in ("train", "val", "test"):
    d = dfs[s]
    lab = d['label'].mean() if 'label' in d.columns else float('nan')
    print(f"{s:5s}: {len(d):>9,} rows   label-rate={lab if lab==lab else float('nan'):.3%}")
    assert not d.duplicated(["station_id", "source", "timestamp"]).any(), \
        f"duplicate timestamps in {s}; refusing ambiguous alignment"
for g in NOAA_GROUPS[:10]:
    sid, src = g.split("|", 1)
    counts = []
    for s in ("train", "val", "test"):
        d = dfs[s]
        counts.append(int(((d.station_id.astype(str) == sid) &
                           (d.source.astype(str) == src)).sum()))
    print("NOAA", g, "split rows", counts)
ev = dfs["events"]
expected_faults = {"spike", "drift", "bias", "stuck", "dropout",
                   "noise_burst", "clipping", "scale_error", "sensor_swap"}
assert not ev.empty and expected_faults <= set(ev["fault"].astype(str)), (
    "injection event log is empty or missing a fault class"
)
print("verified build:", BUILD_ID)
print("\ninjected events:", len(ev))
print(ev.groupby("fault").size())


## 1b · Open-Meteo reference audit (not training labels)

In [ ]:
# Quantify whether each co-located reanalysis stream is useful as a
# spatial/reference signal. These rows stay separate from headline metrics.
reference_rows = []
measured = ["temperature_c", "pressure_hpa", "relative_humidity_pct"]
for om_group in OPENMETEO_GROUPS:
    sid = om_group.split("|", 1)[0]
    for split_name in ("train", "val", "test"):
        om = dfs[split_name]
        om = om[(om.station_id.astype(str) == sid) &
                (om.source.astype(str) == "openmeteo")]
        noaa = dfs[split_name]
        noaa = noaa[(noaa.station_id.astype(str) == sid) &
                    (noaa.source.astype(str) == "noaa_isd")]
        if om.empty or noaa.empty:
            continue
        for col in measured:
            left = noaa[["timestamp", col]].rename(columns={col: "noaa"})
            right = om[["timestamp", col]].rename(columns={col: "reference"})
            joined = left.merge(right, on="timestamp", how="inner")
            joined = joined.replace([np.inf, -np.inf], np.nan).dropna()
            if len(joined) < 24:
                continue
            a, b = joined.noaa.to_numpy(float), joined.reference.to_numpy(float)
            corr = (float(np.corrcoef(a, b)[0, 1])
                    if np.std(a) > 0 and np.std(b) > 0 else float("nan"))
            reference_rows.append({
                "station_id": sid, "split": split_name, "channel": col,
                "noaa_rows": int(len(noaa)), "reference_rows": int(len(om)),
                "overlap_rows": int(len(joined)),
                "overlap_fraction": float(len(joined) / max(1, len(noaa))),
                # Validation/test contain synthetic faults by design; keeping
                # this rate visible prevents the reference audit being mistaken
                # for a clean sensor-agreement estimate.
                "noaa_label_rate": (float(noaa["label"].mean())
                                    if "label" in noaa else 0.0),
                "pearson_r": corr,
                "median_abs_difference": float(np.median(np.abs(a - b))),
            })
reference_report = pd.DataFrame(reference_rows)
reference_report.to_csv(os.path.join(OUT, "openmeteo_reference_report.csv"), index=False)
print("Open-Meteo reference rows:", len(reference_report))
if len(reference_report):
    print(reference_report.groupby("channel")[[
        "overlap_fraction", "pearson_r", "median_abs_difference"
    ]].mean())


## 2 · Train the NOAA headline ensemble (GPU)

Deep heads on `torch`. Early stopping uses a clean chronological tail of the
training period. The station-level holdout set in `splits.json` is excluded
from global model fitting and label-based calibration; its local
climatology/normalisation and POT threshold use clean history only. Test is
never touched during fitting. Open-Meteo groups are intentionally excluded from
this headline run and are used only in the reference audit above.

In [ ]:
from awsad.train_pipeline import run_pipeline

# SMOKE: tiny subset + few epochs to validate the whole pipeline end-to-end
# (featurize -> classical -> deep AEs -> forecaster score_series -> ensemble ->
# eval) in ~10 min on the RTX Pro 6000 before committing to the full run.
NOAA_GROUPS = NOAA_GROUPS[:12]
print(f"SMOKE: restricted to {len(NOAA_GROUPS)} NOAA groups")
DEEP_CFG = {
    "lstm_ae_24":  {"epochs": 2, "batch_size": 512, "windows_per_epoch": 8192,
                    "hidden": 64, "layers": 2, "dropout": 0.10,
                    "balance_groups": True, "loss": "huber", "val_windows": 2048},
    "lstm_ae_168": {"epochs": 2, "batch_size": 256, "windows_per_epoch": 8192,
                    "hidden": 64, "layers": 2, "dropout": 0.10,
                    "balance_groups": True, "loss": "huber", "val_windows": 2048},
    "tx_ae":       {"epochs": 2, "batch_size": 256, "windows_per_epoch": 8192,
                    "d_model": 64, "nhead": 4, "layers": 2, "ff": 128,
                    "bottleneck": 32, "dropout": 0.10,
                    "balance_groups": True, "loss": "huber", "val_windows": 2048},
    "forecaster":  {"epochs": 2, "batch_size": 512, "windows_per_epoch": 8192,
                    "hidden": 64, "layers": 2, "dropout": 0.10,
                    "balance_groups": True, "loss": "huber", "val_windows": 2048},
}
t0 = time.time()

report = run_pipeline(dfs, NOAA_GROUPS, OUT, train_deep=True, deep_cfg=DEEP_CFG)
pathlib.Path(OUT, "training_manifest.json").write_text(json.dumps({
    "seed": SEED, "device": DEVICE, "gpu": GPU_NAME, "gpu_gib": GPU_GB,
    "dataset_build_id": BUILD_ID,
    "dataset_manifest": BUNDLE_MANIFEST,
    "headline_source": "noaa_isd", "headline_groups": NOAA_GROUPS,
    "reference_groups": OPENMETEO_GROUPS, "deep_config": DEEP_CFG,
    "source_metadata": dfs.get("source_metadata", {}),
}, indent=2, default=str))
print(f"\nTOTAL WALL-CLOCK: {(time.time()-t0)/60:.1f} min")


## 3 · Headline metrics (test split)

In [ ]:
t = report["test"]
rows = []
for k, v in t.items():
    if isinstance(v, dict) and "pointwise" in v and "point_adjust" in v:
        rows.append({"detector": k,
                     "pointF1": round(v["pointwise"]["f1"], 4),
                     "pointAdjF1": round(v["point_adjust"]["f1"], 4),
                     "rangeF1": round(v["range_wise"]["f1"], 4),
                     "precision": round(v["pointwise"]["precision"], 4),
                     "recall": round(v["pointwise"]["recall"], 4),
                     "PR_AUC": round(v.get("pr_auc", float("nan")), 4)})
        if "ROC" not in k:
            pass
dfm = pd.DataFrame(rows).sort_values(
    ["pointF1", "rangeF1", "PR_AUC"], ascending=False
)
print(dfm.to_string(index=False))
print("\nPer-fault recall:")
pf = pd.DataFrame(t["per_fault"]).T.sort_index()
print(pf.to_string())
print("\nEvaluation protocol:", json.dumps(report.get("protocol", {}), indent=2))
print("\nEnsemble weights:", json.dumps(t["weights"], indent=None))
print("Thresholds:", json.dumps(t["thresholds"], indent=None, default=str)[:400])


## 4 · Plots

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

sc = np.load(os.path.join(OUT, "scores_cache.npz"))
g = NOAA_GROUPS[0]
y = sc[f"test|{g}|label"]
ens_w = report["test"]["weights"]
comp = [k for k in ens_w if f"test|{g}|{k}" in sc.files]
with open(os.path.join(OUT, "ensemble.json")) as f:
    ens_blob = json.load(f)
norms = ens_blob["per_group_norms"][g]
S = sum(ens_w[k] * ((sc[f"test|{g}|{k}"] - norms[k][0]) /
                    max(norms[k][1] - norms[k][0], 1e-9)) for k in comp)
S = S / max(sum(ens_w[k] for k in comp), 1e-9)

fig, ax = plt.subplots(3, 1, figsize=(15, 9), sharex=False)
ax[0].plot(y, lw=0.7); ax[0].set_title(f"ground-truth labels — {g}")
ax[1].plot(S, lw=0.4, color="darkred"); ax[1].set_title("ensemble score")
for k in comp:
    ax[2].plot(sc[f"test|{g}|{k}"], lw=0.3, label=k)
ax[2].legend(ncol=4, fontsize=7); ax[2].set_title("component scores")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "station_scores.png"), dpi=110)
print("saved station_scores.png")

qc_rep = pd.DataFrame(report["test"]["ablation_loo"], index=["loo_pointF1"]).T
print(qc_rep)


## 5 · External benchmark — Numenta Anomaly Benchmark (NAB)

Independent, labeled, streaming-anomaly corpus (transportable method: a small
univariate Isolation Forest because the station models are weather-specific).
The detector and threshold are fitted on the leading training prefix only.

In [ ]:
from awsad.evaluation.nab import evaluate_nab
try:
    nab_df = evaluate_nab(os.path.join(NAB_DIR))
    print(nab_df.to_string(index=False))
    nab_df.to_csv(os.path.join(OUT, "nab_results.csv"), index=False)
except Exception as e:
    print("NAB evaluation skipped:", e)


## 6 · Export artifacts (models, thresholds, scores, metrics)

In [ ]:
import zipfile
zip_path = os.path.join(WORKDIR, "skyguard_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in glob.glob(os.path.join(OUT, "**", "*"), recursive=True):
        if os.path.isfile(f):
            zf.write(f, os.path.relpath(f, OUT))
size_mb = os.path.getsize(zip_path) / 1e6
print(f"artifacts -> {zip_path} ({size_mb:.1f} MB)")
print("Download from the notebook Output tab; unpack into the repo's artifacts/ "
      "directory to power the Streamlit dashboard (streamlit run app/streamlit_app.py).")
